# RMT · Remember 기능 점검
원본 RMT를 사용하며, 기능 점검용으로 서로 다른 key/value와 context별 모든 질의를 학습합니다.
train/validation/test context는 서로 겹치지 않습니다. curriculum은 `1 → 2 → 3쌍`입니다.
T4 · D128/4층/memory32 · effective batch1024 · FP32.
각 단계의 생성 exact match와 모든 질의 성공률이 모두99% 이상이면 진행합니다.
최대2,900 update · 학습 wall budget35분, 최종 test는 별도입니다. 전체 논문 실험 재현과 구분합니다.


In [ ]:
from pathlib import Path
import sys, subprocess, torch
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/GRT')
RUN_DIR = Path('/content/drive/MyDrive/GRT/runs/rmt_remember_functional_t4_seed1234')
assert (REPO_DIR / 'configs/rmt_remember_functional.yaml').exists(), '수정본을 pull한 checkout이 필요합니다.'
assert torch.cuda.is_available(), 'T4 GPU runtime을 선택하세요.'
print('Python:', sys.version.split()[0])
installation = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)],
                              text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('패키지 설치 실패: 위 pip 오류를 확인하세요.')
print('Package installation completed.')
sys.path.insert(0, str(REPO_DIR / 'src'))
from grt.config import load_config
print(torch.cuda.get_device_name(0), 'VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('run:', RUN_DIR)


In [ ]:
# 동일 명령을 다시 실행하면 완료된 단계를 건너뛰고 해당 단계 last.pt에서 재개합니다.
subprocess.run([sys.executable, '-u', 'scripts/train.py',
                '--config', 'configs/rmt_paper_remember.yaml', 'configs/rmt_remember_functional.yaml',
                '--output-dir', str(RUN_DIR), '--device', 'cuda'], cwd=REPO_DIR, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.loads((RUN_DIR / 'summary.json').read_text())
evaluation = json.loads((RUN_DIR / 'evaluation.json').read_text())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
offset = 0
for stage in summary['stages']:
    # Use the current RUN_DIR even if Drive mount paths changed.
    stage_dir = RUN_DIR / Path(stage['checkpoint']).parent.name
    rows = [json.loads(line) for line in (stage_dir / 'metrics.jsonl').read_text().splitlines()]
    rows = sorted({r['global_step']: r for r in rows if 'val/loss' in r}.values(), key=lambda r:r['global_step'])
    steps = [offset + r['global_step'] for r in rows]
    label = str(stage['num_pairs']) + ' pairs'
    axes[0].plot(steps, [r['val/loss'] for r in rows], label=label)
    axes[1].plot(steps, [r['val/autoregressive/all_queries_exact_match'] for r in rows], label=label + ' all queries')
    axes[1].plot(steps, [r['val/autoregressive/value_exact_match'] for r in rows], linestyle='--', label=label + ' value')
    offset += stage['global_step']
axes[0].set_ylabel('validation CE (value + EOS)')
axes[1].set_ylabel('generated accuracy / all-query success')
axes[1].axhline(.99, color='gray', linestyle='--')
axes[1].set_ylim(0, 1.02)
for ax in axes:
    ax.set_xlabel('total optimizer updates')
    ax.legend()
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR / 'convergence.png', dpi=150)
plt.show()
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(json.dumps(evaluation, indent=2, ensure_ascii=False))
print('공유: summary.json, evaluation.json, convergence.png 및 각 stage의 metrics.jsonl, resolved_config.yaml, metadata.json')
